## Step 3: Verifying our clean dataset
Before proceeding with the MLP training steps, it is essential to double-check the clean dataset.

### Step 3.1: Verifying missing values
First, we will check if there are any null values ​​in any column of our dataset.

In [ ]:
total_missing = clean_df.isnull().sum().sum()
assert total_missing == 0, \
    f" TEST FAILED: {total_missing} missing values remain."
print("PASS: no missing values")

### Step 3.2: Verifying our target column values
Finally, let's check if our target column only contains expected values.
We will consider allowed_values ​​as 0, 1, 2, 3, 4, and 5, which correspond to the possible failure type (after encoding).

In [ ]:
invalid_mask = ~df[target_col].isin(allowed_values)
invalid_count = invalid_mask.sum()
assert invalid_count == 0 , \
    f"TEST FAILED: {invalid_count} unexpected values in '{target_col}':" \
    f"{df.loc[invalid_mask, target_col].unique()}"
print (f"PASS: '{ target_col }' contains only valid values.")

## Step 4: Splitting

### Step 4.1: Selecting our features
After Feature Selection, we must change the value of X so that it only stores the variables that we will use in training our MLP.

In [ ]:
selected_features = ["Air temperature [K]", "Torque [Nm]", "Tool wear [min]"]
X = X[selected_features]

### Step 4.2: Splitting
Splitting plays the primary role of separating the data for testing and training. In our case, 20% of the data will be used for testing. However, to do this, stratification must also be performed, which ensures that the proportion of classes is maintained during this splitting stage.

In [ ]:
def split_train_test (X, y, test_size: float = 0.2 , random_state : int = 42):
    X_train , X_test , y_train , y_test = train_test_split(X, y,
    test_size = test_size,
    random_state = random_state,
    stratify = y                # preserves class proportions
    )
    train_df = pd.concat([X_train, y_train], axis = 1)
    test_df = pd.concat([X_test, y_test], axis = 1)
    return train_df, test_df

## Step 5: Preparing our dataloaders

### Step 5.1: Separation
It is crucial to separate the columns that can be selected during the Feature Selection process from the target column. This way, to avoid data leakage in the future, we will insert the non-target columns into the variable y and the target column into the variable X.

In [ ]:
def prepare_dataloaders (train_df, test_df, target_col: str, batch_size: int):

    # Separate features and labels
    X_train = train_df.drop(columns = [target_col]).values.astype(np.float32)
    y_train = train_df[target_col].values.astype(np.int64)

    X_test = test_df.drop(columns = [target_col]).values.astype(np.float32)
    y_test = test_df[target_col].values.astype(np.int64)

### Step 5.2: Scale Normalization
It is crucial that the numerical columns are entered on similar scales to reduce instability during the training process, as well as to minimize any type of bias.

Therefore, we will do this using the StandardScaler, which utilizes the Z-score standardization mechanism, where each column must assume a mean of 0 and a standard deviation of 1.

**The fit should not be applied to the test data, as this would lead to data leakage. Therefore, only a simple transform should be applied to the test data.**

In [ ]:
    # Standardize : fit on train , transform both
    scaler = StandardScaler ()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test) # NO fit here !

### Step 5.3: Creating ours tensors
In order to use PyTorch, we must convert our data into tensors, and then generate a dataset with these tensors.


In [ ]:
    # Build TensorDatasets
    train_ds = TensorDataset(torch.tensor(X_train), torch.tensor(y_train ))
    test_ds = TensorDataset(torch.tensor(X_test), torch.tensor(y_test ))

### Step 5.4: Generating ours mini-batches
Finally, it is crucial to perform the Dataloader operation on our tensor dataset, so that we can divide our data into small batches, as well as shuffle the data to avoid order bias.

In [ ]:
    # Wrap in DataLoaders
    train_loader = DataLoader(train_ds, batch_size = batch_size, shuffle = True)
    test_loader = DataLoader(test_ds, batch_size = batch_size, shuffle = False)

    return train_loader, test_loader, scaler

## Step 6: MLP training

### Step 6.1: Training setup
First, we define the basic parameters for the operation of our MLP.

* **device:** allows the selection of the device for running our training (CPU or GPU)
* **model:** creation of the MLP model
* **criterion:** definition of the loss function
* **optimizer:** updates the learning rate based on the error
* **best_val_loss:** starts with an infinite value and is used to find the best possible model
* **early stopping:** has the crucial function of stopping the training when it verifies that the model stops improving.

In [ ]:
def train_model(config : dict, train_loader, test_loader, input_dim: int) -> nn.Module:

    # --- Setup ---
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = MLP(input_dim = input_dim, hidden_sizes = config ["model"]["hidden_sizes"], dropout = config ["model"]["dropout"]).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr = config["model"]["learning_rate"])
    best_val_loss = float("inf")
    
    # --- Early stopping setup ---
    patience = config["model"]["early_stopping_patience"]
    patience_counter = 0

### Step 6.2: Training phase
This is the stage where the actual learning occurs. The basic learning stage is as follows: 

*clear gradients --> prediction --> error calculation --> compute gradients --> update weights*

In [ ]:
    for epoch in range(config ["model"]["epochs"]):

        # Training phase
        model.train()        # activates Dropout
        train_loss = 0.0

        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()        # clear previous gradients
            output = model(X_batch)      # forward pass
            loss = criterion(output, y_batch)
            loss.backward()              # compute gradients
            optimizer.step()             # update weights

            train_loss += loss.item() * X_batch.size(0)

        train_loss /= len(train_loader.dataset)

### Step 6.3: Validation phase
At this phase, the steps are practically the same as in the training phase, with the crucial difference being that there is no actual learning, as we are simulating the input of new data.

In [ ]:
       # Validation phase
        model.eval()        # deactivates Dropout
        val_loss = 0.0
        y_true = []
        y_pred = []
        with torch.no_grad():       # no gradient computation
            for X_batch, y_batch in test_loader:
                X_batch = X_batch.to(device)
                y_batch = y_batch.to(device)

                output = model(X_batch)
                loss = criterion(output, y_batch)
                val_loss += loss.item() * X_batch.size(0)

                pred = torch.argmax(output, dim=1)
                y_true.extend(y_batch.cpu().numpy())
                y_pred.extend(pred.cpu().numpy())

        val_loss /= len(test_loader.dataset)

### Step 6.4: Classification metrics
We use metrics to evaluate the performance of our model.
* **Accuracy:** measures the percentage of cases in which the model was correct.
* **Precision:** it measures the number of cases in which the model correctly classified them as positive.
* **Recall:** it measures the number of truly positive cases that the model found.
* **F1-Score:** it measures an average between the Precision and Recall values.
* **Confusion Matrix:** it is crucial for us to compare the actual values ​​with the values ​​predicted by the model.

In [ ]:
        
        val_acc = accuracy_score(y_true, y_pred)
        val_prec = precision_score(y_true, y_pred, average="weighted")
        val_rec = recall_score(y_true, y_pred, average="weighted")
        val_f1 = f1_score(y_true, y_pred, average="weighted")
        cm = confusion_matrix(y_true, y_pred)
        plt.figure(figsize=(6,5))
        sns.heatmap(cm, annot=True, fmt="d")

### Step 6.5: Saving our model

In [ ]:
        # Early stopping
        if val_loss < best_val_loss :
            best_val_loss = val_loss
            patience_counter = 0

            # Save the best model checkpoint
            torch.save(model.state_dict(), "best_model.pt")

            # Log checkpoint as a W&B artifact
            model_artifact = wandb.Artifact ("trained_model", type = "model", description = f" MLP checkpoint at epoch { epoch }")
            model_artifact.add_file("best_model.pt")
            wandb.log_artifact(model_artifact)

        else :
            patience_counter += 1

            if patience_counter >= patience:
                print(f"Early stopping at epoch {epoch}." f"Best val_loss: {best_val_loss:.4f}")
                break

    # Load the best weights before returning
    model.load_state_dict(torch.load("best_model.pt"))
    return model